# Set Up

In [1]:
from google.colab import drive, userdata
import os, json, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import ast
from datetime import datetime
from sklearn.metrics import accuracy_score, f1_score

drive.mount('/content/drive')

try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except:
    pass

BASE    = "/content/drive/MyDrive/rare_disease_project"
DATA    = f"{BASE}/data"
RESULTS = f"{BASE}/results"
MODELS  = f"{BASE}/models"

device = torch.device("cuda" if torch.cuda.is_available()
                      else "cpu")
print(f"✓ Device: {torch.cuda.get_device_name(0)}")

# ── Load tier_a splits ─────────────────────────────────────────
with open(f"{DATA}/splits/tier_a_train.pkl", "rb") as f:
    raw_train = pickle.load(f)
with open(f"{DATA}/splits/tier_a_test.pkl", "rb") as f:
    raw_test = pickle.load(f)

print(f"✓ Tier-A loaded")
print(f"  Raw train : {len(raw_train)}")
print(f"  Raw test  : {len(raw_test)}")
print(f"  Columns   : {list(raw_train.columns)}")
print(f"\n  Train label sample: "
      f"{sorted(raw_train['label'].unique())[:5]}")
print(f"  Test label sample : "
      f"{sorted(raw_test['label'].unique())[:5]}")

Mounted at /content/drive
✓ Device: Tesla T4
✓ Tier-A loaded
  Raw train : 10711
  Raw test  : 1941
  Columns   : ['orpha_code', 'case_id', 'disease_name', 'symptoms', 'num_symptoms', 'images', 'num_images', 'age', 'gender', 'symptom_text', 'label']

  Train label sample: [np.int64(7), np.int64(37), np.int64(55), np.int64(56), np.int64(62)]
  Test label sample : [np.int64(15), np.int64(110), np.int64(141), np.int64(155), np.int64(180)]


# Data Preparation (Fixed Label Mapping)

In [2]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split

# Load the tiers to get tier_a disease codes
with open(f"{DATA}/clean_tiers.pkl", "rb") as f:
    tiers = pickle.load(f)

print(f"Tier keys: {list(tiers.keys())}")
print(f"Tier A codes (first 5): {tiers['tier_a'][:5]}")
print(f"Tier A count: {len(tiers['tier_a'])}")

# Load full dataset
df_full = pd.read_csv(f"{DATA}/clean_multimodal_samples.csv")
print(f"\nFull dataset: {df_full.shape}")

# Filter to tier_a diseases only using orpha_code
tier_a_codes = [str(c) for c in tiers['tier_a']]
df_tier_a = df_full[
    df_full['orpha_code'].astype(str).isin(tier_a_codes)
].reset_index(drop=True)

print(f"Tier-A samples : {len(df_tier_a)}")
print(f"Tier-A diseases: {df_tier_a['disease_name'].nunique()}")

# Encode labels
from sklearn.preprocessing import LabelEncoder
new_le = LabelEncoder()
df_tier_a['label'] = new_le.fit_transform(
    df_tier_a['disease_name'])

# Add symptom_text
def make_symptom_text(sym_str):
    try:
        import ast
        syms = sym_str if isinstance(sym_str, list) \
                       else ast.literal_eval(sym_str)
        return ' [SEP] '.join(
            [s.lower().strip() for s in syms])
    except:
        return str(sym_str)

df_tier_a['symptom_text'] = df_tier_a['symptoms'].apply(
    make_symptom_text)

# Filter classes with >= 2 samples
label_counts = df_tier_a['label'].value_counts()
valid        = label_counts[label_counts >= 2].index
df_tier_a    = df_tier_a[
    df_tier_a['label'].isin(valid)
].reset_index(drop=True)

# 80/20 split — stratified
train_df, test_df = train_test_split(
    df_tier_a, test_size=0.2,
    random_state=42,
    stratify=df_tier_a['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Remap labels 0..N
all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old
                 in enumerate(all_labels)}
reverse_remap = {new: old for old, new
                 in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(
    subset=['label']).reset_index(drop=True)
test_df['label'] = test_df['label'].astype(int)

print(f"\n✓ Tier-A split created correctly")
print(f"  Train   : {len(train_df)}")
print(f"  Test    : {len(test_df)}")
print(f"  Classes : {NUM_CLASSES}")
print(f"\n  Train label range: "
      f"{train_df['label'].min()} → "
      f"{train_df['label'].max()}")
print(f"  Test label range : "
      f"{test_df['label'].min()} → "
      f"{test_df['label'].max()}")

Tier keys: ['tier_a', 'tier_b', 'tier_c', 'counts']
Tier A codes (first 5): ['95507', '544', '547', '135', '877']
Tier A count: 88

Full dataset: (36487, 9)
Tier-A samples : 17123
Tier-A diseases: 88

✓ Tier-A split created correctly
  Train   : 13698
  Test    : 3425
  Classes : 88

  Train label range: 0 → 87
  Test label range : 0 → 87


# Improved NLP Training

In [3]:
from google.colab import drive, userdata
import os, json, pickle, ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

drive.mount('/content/drive')

BASE    = "/content/drive/MyDrive/rare_disease_project"
DATA    = f"{BASE}/data"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"

device = torch.device("cuda" if torch.cuda.is_available()
                      else "cpu")
print(f"✓ Device: {torch.cuda.get_device_name(0)}")

# ── Load tier_a data properly ──────────────────────────────────
with open(f"{DATA}/clean_tiers.pkl", "rb") as f:
    tiers = pickle.load(f)

df_full = pd.read_csv(f"{DATA}/clean_multimodal_samples.csv")

def make_symptom_text(sym_str):
    try:
        syms = sym_str if isinstance(sym_str, list) \
                       else ast.literal_eval(sym_str)
        return ' [SEP] '.join(
            [s.lower().strip() for s in syms])
    except:
        return str(sym_str)

df_full['symptom_text'] = df_full['symptoms'].apply(
    make_symptom_text)

new_le = LabelEncoder()
df_full['label'] = new_le.fit_transform(
    df_full['disease_name'])

# Filter to Tier-A only
tier_a_codes = [str(c) for c in tiers['tier_a']]
df_tier_a    = df_full[
    df_full['orpha_code'].astype(str).isin(tier_a_codes)
].reset_index(drop=True)

print(f"✓ Tier-A samples : {len(df_tier_a)}")
print(f"  Diseases       : {df_tier_a['disease_name'].nunique()}")

# Filter classes with >=2 samples
label_counts = df_tier_a['label'].value_counts()
valid        = label_counts[label_counts >= 2].index
df_tier_a    = df_tier_a[
    df_tier_a['label'].isin(valid)
].reset_index(drop=True)

# 80/20 stratified split
train_df, test_df = train_test_split(
    df_tier_a, test_size=0.2,
    random_state=42,
    stratify=df_tier_a['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Remap labels
all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old
                 in enumerate(all_labels)}
reverse_remap = {new: old for old, new
                 in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(
    subset=['label']).reset_index(drop=True)
test_df['label']  = test_df['label'].astype(int)

print(f"✓ Data ready")
print(f"  Train   : {len(train_df)}")
print(f"  Test    : {len(test_df)}")
print(f"  Classes : {NUM_CLASSES}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Device: Tesla T4
✓ Tier-A samples : 17123
  Diseases       : 88
✓ Data ready
  Train   : 13698
  Test    : 3425
  Classes : 88


# Text Augmentation + Dataset

In [4]:
import random

def augment_text(text, n=3):
    """Shuffle symptom order for augmentation"""
    try:
        syms = text.split(' [SEP] ')
        augmented = []
        for _ in range(n):
            shuffled = syms.copy()
            random.shuffle(shuffled)
            augmented.append(' [SEP] '.join(shuffled))
        return augmented
    except:
        return []

# Augment minority classes
class_counts   = train_df['label'].value_counts()
minority_labels = class_counts[class_counts <= 5].index.tolist()

aug_rows = []
for _, row in train_df.iterrows():
    if row['label'] in minority_labels:
        for aug_text in augment_text(
                row['symptom_text'], n=5):
            new_row = row.copy()
            new_row['symptom_text'] = aug_text
            aug_rows.append(new_row)

if aug_rows:
    aug_df   = pd.DataFrame(aug_rows)
    train_aug = pd.concat(
        [train_df, aug_df]).reset_index(drop=True)
else:
    train_aug = train_df.copy()

print(f"✓ Augmented train: {len(train_aug)} "
      f"(+{len(aug_rows)} synthetic rows)")

# ── Tokenizer + Dataset ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2")

class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts      = df['symptom_text'].tolist()
        self.labels     = df['label'].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(
                                self.labels[idx],
                                dtype=torch.long)
        }

train_ds = SymptomDataset(train_aug, tokenizer)
test_ds  = SymptomDataset(test_df,   tokenizer)

train_loader = DataLoader(
    train_ds, batch_size=32,
    shuffle=True,  num_workers=2)
test_loader  = DataLoader(
    test_ds,  batch_size=32,
    shuffle=False, num_workers=2)

print(f"✓ DataLoaders ready")
print(f"  Train : {len(train_ds)} | Batches: {len(train_loader)}")
print(f"  Test  : {len(test_ds)}  | Batches: {len(test_loader)}")

✓ Augmented train: 13698 (+0 synthetic rows)


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✓ DataLoaders ready
  Train : 13698 | Batches: 429
  Test  : 3425  | Batches: 108


# Improved Model (Deeper Head + Label Smoothing)

In [5]:
class BioBERTClassifierV2(nn.Module):
    """Improved version with deeper head + layer norm"""
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(
            "dmis-lab/biobert-base-cased-v1.2")

        # Unfreeze last 4 layers for better fine-tuning
        for i, layer in enumerate(
                self.bert.encoder.layer):
            if i >= 8:  # unfreeze layers 8-11
                for param in layer.parameters():
                    param.requires_grad = True
            else:
                for param in layer.parameters():
                    param.requires_grad = False

        hidden = self.bert.config.hidden_size  # 768

        # Deeper classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(dropout),
            nn.Linear(hidden, 768),
            nn.GELU(),
            nn.LayerNorm(768),
            nn.Dropout(dropout),
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask)
        # Mean pooling instead of just CLS token
        mask  = attention_mask.unsqueeze(-1).float()
        token = out.last_hidden_state
        mean  = (token * mask).sum(1) / mask.sum(1)
        return self.classifier(mean)


# Label smoothing loss — better generalization
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.cls       = classes

    def forward(self, pred, target):
        confidence = 1.0 - self.smoothing
        smooth_val = self.smoothing / (self.cls - 1)
        one_hot    = torch.full_like(
            pred, smooth_val)
        one_hot.scatter_(1, target.unsqueeze(1),
                         confidence)
        log_prob = nn.functional.log_softmax(pred, dim=1)
        return -(one_hot * log_prob).sum(dim=1).mean()


# ── Training setup ─────────────────────────────────────────────
EPOCHS     = 20
model_v2   = BioBERTClassifierV2(NUM_CLASSES).to(device)
criterion  = LabelSmoothingLoss(NUM_CLASSES, smoothing=0.1)
optimizer  = AdamW(
    model_v2.parameters(),
    lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 8,
    num_training_steps=total_steps
)

trainable = sum(p.numel() for p in model_v2.parameters()
                if p.requires_grad)
print(f"✓ Model V2 built")
print(f"  Trainable params : {trainable:,}")
print(f"  Improvement over V1:")
print(f"    - Mean pooling instead of CLS only")
print(f"    - Deeper 3-layer head")
print(f"    - Last 4 BERT layers unfrozen")
print(f"    - Label smoothing loss")
print(f"    - Text augmentation for minority classes")

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model V2 built
  Trainable params : 52,639,832
  Improvement over V1:
    - Mean pooling instead of CLS only
    - Deeper 3-layer head
    - Last 4 BERT layers unfrozen
    - Label smoothing loss
    - Text augmentation for minority classes


# Train + Evaluate

In [6]:
import matplotlib.pyplot as plt

losses = []
accs   = []

print(f"\nTraining Improved NLP — {EPOCHS} epochs")
print(f"  Samples : {len(train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

best_acc   = 0
best_epoch = 0

for epoch in range(EPOCHS):
    model_v2.train()
    total_loss, correct, total = 0, 0, 0

    for batch in train_loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model_v2(ids, mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model_v2.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    acc      = correct / total * 100
    losses.append(avg_loss)
    accs.append(acc)

    if acc > best_acc:
        best_acc   = acc
        best_epoch = epoch + 1
        # Save best checkpoint
        torch.save(model_v2.state_dict(),
                   '/tmp/best_nlp_v2.pt')

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%"
          + (" ← best" if epoch+1 == best_epoch else ""))

print("-" * 55)
print(f"✓ Training complete | Best epoch: {best_epoch}")

# Load best model for evaluation
model_v2.load_state_dict(
    torch.load('/tmp/best_nlp_v2.pt'))
model_v2.eval()

# Evaluate
all_preds, all_labels_list = [], []
topk_correct = {1: 0, 3: 0, 5: 0}
total = 0

with torch.no_grad():
    for batch in test_loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        logits = model_v2(ids, mask)

        preds  = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels_list.extend(labels.cpu().numpy())

        for k_val in [1, 3, 5]:
            k_act = min(k_val, logits.size(1))
            topk  = logits.topk(k_act, dim=1).indices
            for i, lbl in enumerate(labels):
                if lbl in topk[i]:
                    topk_correct[k_val] += 1
        total += labels.size(0)

metrics = {
    "accuracy"     : round(accuracy_score(
                        all_labels_list, all_preds)*100, 2),
    "f1_macro"     : round(f1_score(
                        all_labels_list, all_preds,
                        average='macro',
                        zero_division=0)*100, 2),
    "top1_accuracy": round(topk_correct[1]/total*100, 2),
    "top3_accuracy": round(topk_correct[3]/total*100, 2),
    "top5_accuracy": round(topk_correct[5]/total*100, 2),
    "total_samples": total
}

print("\n" + "=" * 55)
print("IMPROVED NLP V2 — RESULTS")
print("=" * 55)
print(f"  Accuracy     : {metrics['accuracy']}%"
      f"  (was 16.89%)")
print(f"  F1 Macro     : {metrics['f1_macro']}%")
print(f"  Top-1 Acc    : {metrics['top1_accuracy']}%")
print(f"  Top-3 Acc    : {metrics['top3_accuracy']}%")
print(f"  Top-5 Acc    : {metrics['top5_accuracy']}%")
print(f"  Test samples : {metrics['total_samples']}")
print(f"\n  Improvement  : "
      f"{metrics['accuracy']-16.89:+.2f}% over baseline")


Training Improved NLP — 20 epochs
  Samples : 13698
  Classes : 88
-------------------------------------------------------


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Epoch 01/20 | Loss: 4.4473 | Train Acc: 2.35% ← best
Epoch 02/20 | Loss: 4.1876 | Train Acc: 8.13% ← best
Epoch 03/20 | Loss: 3.8004 | Train Acc: 15.32% ← best
Epoch 04/20 | Loss: 3.4926 | Train Acc: 21.78% ← best
Epoch 05/20 | Loss: 3.2773 | Train Acc: 26.73% ← best
Epoch 06/20 | Loss: 3.1186 | Train Acc: 30.92% ← best
Epoch 07/20 | Loss: 2.9904 | Train Acc: 33.91% ← best
Epoch 08/20 | Loss: 2.8829 | Train Acc: 37.56% ← best
Epoch 09/20 | Loss: 2.7743 | Train Acc: 40.35% ← best
Epoch 10/20 | Loss: 2.6899 | Train Acc: 42.47% ← best
Epoch 11/20 | Loss: 2.6063 | Train Acc: 44.50% ← best
Epoch 12/20 | Loss: 2.5268 | Train Acc: 46.50% ← best
Epoch 13/20 | Loss: 2.4646 | Train Acc: 48.47% ← best
Epoch 14/20 | Loss: 2.3934 | Train Acc: 50.42% ← best
Epoch 15/20 | Loss: 2.3387 | Train Acc: 51.57% ← best
Epoch 16/20 | Loss: 2.2934 | Train Acc: 52.88% ← best
Epoch 17/20 | Loss: 2.2488 | Train Acc: 54.56% ← best
Epoch 18/20 | Loss: 2.2214 | Train Acc: 55.50% ← best
Epoch 19/20 | Loss: 2.1978 | T

# Save to Drive

In [7]:
import json
from datetime import datetime

# Save model
torch.save({
    'model_state_dict': model_v2.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : metrics,
    'architecture'    : 'BioBERTClassifierV2',
    'improvements'    : [
        'Mean pooling',
        'Deeper 3-layer head',
        'Last 4 BERT layers unfrozen',
        'Label smoothing loss',
        'Text augmentation'
    ]
}, f"{MODELS}/improved_nlp_v2.pt")

print(f"✓ Model saved: improved_nlp_v2.pt")

# Save summary
summary = {
    "timestamp" : str(datetime.now()),
    "model"     : "BioBERTClassifierV2",
    "dataset"   : "ZebraMap Tier-A",
    "epochs"    : EPOCHS,
    "metrics"   : metrics,
    "baseline"  : {"accuracy": 16.89, "top5": 36.39},
    "gain"      : round(metrics['accuracy'] - 16.89, 2)
}
with open(f"{RESULTS}/nlp_v2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"✓ Summary saved")
print(f"\nNext steps:")
print(f"  1. Download improved_nlp_v2.pt from Drive to PC")
print(f"  2. Re-save with clean numpy (resave script)")
print(f"  3. Upload to HuggingFace")
print(f"  4. Update model_loader.py")
print(f"  5. Redeploy on HuggingFace Spaces")

✓ Model saved: improved_nlp_v2.pt
✓ Summary saved

Next steps:
  1. Download improved_nlp_v2.pt from Drive to PC
  2. Re-save with clean numpy (resave script)
  3. Upload to HuggingFace
  4. Update model_loader.py
  5. Redeploy on HuggingFace Spaces


#  NLP Dataset + Model

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2")
print("✓ Tokenizer loaded")

class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts      = df['symptom_text'].tolist()
        self.labels     = df['label'].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(
                                self.labels[idx],
                                dtype=torch.long)
        }

class BioBERTClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(
            "dmis-lab/biobert-base-cased-v1.2")
        hidden = self.bert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask)
        return self.classifier(
            out.last_hidden_state[:, 0, :])

nlp_train_ds = SymptomDataset(train_df, tokenizer)
nlp_test_ds  = SymptomDataset(test_df,  tokenizer)

nlp_train_loader = DataLoader(
    nlp_train_ds, batch_size=32,
    shuffle=True,  num_workers=2)
nlp_test_loader  = DataLoader(
    nlp_test_ds,  batch_size=32,
    shuffle=False, num_workers=2)

print(f"✓ DataLoaders ready")
print(f"  Train : {len(nlp_train_ds)} samples "
      f"| {len(nlp_train_loader)} batches")
print(f"  Test  : {len(nlp_test_ds)} samples "
      f"| {len(nlp_test_loader)} batches")
print(f"  Classes: {NUM_CLASSES}")

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✓ Tokenizer loaded
✓ DataLoaders ready
  Train : 13698 samples | 429 batches
  Test  : 3425 samples | 108 batches
  Classes: 88


# Train NLP (20 Epochs)

In [ ]:
# ── Improvement: 20 epochs + class weights ─────────────────────
NLP_EPOCHS  = 20
criterion   = nn.CrossEntropyLoss()
nlp_model   = BioBERTClassifier(NUM_CLASSES).to(device)
optimizer   = AdamW(nlp_model.parameters(),
                    lr=2e-5, weight_decay=0.01)
total_steps = len(nlp_train_loader) * NLP_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

nlp_losses = []
nlp_accs   = []

print(f"Training NLP Tier-A — {NLP_EPOCHS} epochs")
print(f"  Samples : {len(nlp_train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

for epoch in range(NLP_EPOCHS):
    nlp_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in nlp_train_loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = nlp_model(ids, mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            nlp_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(nlp_train_loader)
    acc      = correct / total * 100
    nlp_losses.append(avg_loss)
    nlp_accs.append(acc)
    print(f"Epoch {epoch+1:02d}/{NLP_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ NLP training complete")

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training NLP Tier-A — 20 epochs
  Samples : 13698
  Classes : 88
-------------------------------------------------------
Epoch 01/20 | Loss: 4.4334 | Train Acc: 2.41%
Epoch 02/20 | Loss: 4.0478 | Train Acc: 9.69%
Epoch 03/20 | Loss: 3.5213 | Train Acc: 18.89%
Epoch 04/20 | Loss: 3.1376 | Train Acc: 26.27%
Epoch 05/20 | Loss: 2.8524 | Train Acc: 31.72%
Epoch 06/20 | Loss: 2.5933 | Train Acc: 37.06%
Epoch 07/20 | Loss: 2.3639 | Train Acc: 41.15%
Epoch 08/20 | Loss: 2.1575 | Train Acc: 46.44%
Epoch 09/20 | Loss: 1.9613 | Train Acc: 50.47%
Epoch 10/20 | Loss: 1.7826 | Train Acc: 54.32%
Epoch 11/20 | Loss: 1.6291 | Train Acc: 58.16%
Epoch 12/20 | Loss: 1.4830 | Train Acc: 61.77%
Epoch 13/20 | Loss: 1.3526 | Train Acc: 64.99%
Epoch 14/20 | Loss: 1.2425 | Train Acc: 67.70%
Epoch 15/20 | Loss: 1.1410 | Train Acc: 70.56%
Epoch 16/20 | Loss: 1.0582 | Train Acc: 72.59%
Epoch 17/20 | Loss: 0.9847 | Train Acc: 74.41%
Epoch 18/20 | Loss: 0.9218 | Train Acc: 75.97%
Epoch 19/20 | Loss: 0.8881 | Train 

# Evaluate NLP

In [ ]:
def evaluate_topk(model, loader, device,
                  is_cnn=False, k=5):
    model.eval()
    all_preds, all_labels = [], []
    topk_correct = {1: 0, 3: 0, 5: 0}
    total = 0

    with torch.no_grad():
        for batch in loader:
            if is_cnn:
                inputs = batch['image'].to(device)
                labels = batch['label'].to(device)
                logits = model(inputs)
            else:
                ids    = batch['input_ids'].to(device)
                mask   = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                logits = model(ids, mask)

            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for k_val in [1, 3, 5]:
                k_act = min(k_val, logits.size(1))
                topk  = logits.topk(k_act, dim=1).indices
                for i, lbl in enumerate(labels):
                    if lbl in topk[i]:
                        topk_correct[k_val] += 1
            total += labels.size(0)

    if total == 0:
        print("⚠ No test samples found!")
        return {}

    return {
        "accuracy"     : round(accuracy_score(
                            all_labels, all_preds)*100, 2),
        "f1_macro"     : round(f1_score(
                            all_labels, all_preds,
                            average='macro',
                            zero_division=0)*100, 2),
        "f1_weighted"  : round(f1_score(
                            all_labels, all_preds,
                            average='weighted',
                            zero_division=0)*100, 2),
        "top1_accuracy": round(
            topk_correct[1]/total*100, 2),
        "top3_accuracy": round(
            topk_correct[3]/total*100, 2),
        "top5_accuracy": round(
            topk_correct[5]/total*100, 2),
        "total_samples": total
    }

print("Evaluating NLP Tier-A...")
nlp_metrics = evaluate_topk(
    nlp_model, nlp_test_loader, device)

print("\n" + "=" * 55)
print("IMPROVED DAY 7 — NLP TIER-A RESULTS")
print("=" * 55)
for k, v in nlp_metrics.items():
    print(f"  {k:<20}: {v}")

torch.save({
    'model_state_dict': nlp_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : nlp_metrics
}, f"{MODELS}/improved_nlp_tier_a.pt")
print(f"\n✓ NLP Tier-A model saved")

Evaluating NLP Tier-A...

IMPROVED DAY 7 — NLP TIER-A RESULTS
  accuracy            : 31.82
  f1_macro            : 31.89
  f1_weighted         : 31.29
  top1_accuracy       : 31.82
  top3_accuracy       : 51.94
  top5_accuracy       : 60.35
  total_samples       : 3425

✓ NLP Tier-A model saved


# Train + Evaluate CNN Tier-A

In [ ]:
import torchvision.models as models
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import os
from PIL import Image

# ================= DATASET =================
class ZebraMapImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.samples   = []
        self.transform = transform

        for _, row in df.iterrows():
            try:
                imgs = row['images'] if isinstance(row['images'], list) else eval(row['images'])
                for img_info in imgs:
                    if os.path.exists(img_info['path']):
                        self.samples.append({
                            'path' : img_info['path'],
                            'label': row['label']
                        })
                        break
            except:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        try:
            img = Image.open(sample['path']).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)

        return {
            'image': img,
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

# ================= TRANSFORMS =================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ================= DATA =================
print("Building CNN Tier-A datasets...")
cnn_train_ds = ZebraMapImageDataset(train_df, train_transform)
cnn_test_ds  = ZebraMapImageDataset(test_df,  test_transform)

cnn_train_loader = DataLoader(
    cnn_train_ds,
    batch_size=32,   # go back to 32 (64 is slowing CPU)
    shuffle=True,
    num_workers=4,   # increase workers
    pin_memory=True
)

cnn_test_loader = DataLoader(
    cnn_test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"✓ CNN DataLoaders ready")
print(f"  Train : {len(cnn_train_ds)}")
print(f"  Test  : {len(cnn_test_ds)}")

# ================= MODEL =================
class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1)

        for layer in list(backbone.children())[:-3]:
            for param in layer.parameters():
                param.requires_grad = False

        self.features = nn.Sequential(*list(backbone.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# ================= TRAIN SETUP =================
CNN_EPOCHS = 10   # ✅ changed (20 → 10)

criterion = nn.CrossEntropyLoss()
cnn_model  = ResNet50Classifier(NUM_CLASSES).to(device)

optimizer2 = AdamW(
    filter(lambda p: p.requires_grad, cnn_model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

scheduler2 = CosineAnnealingLR(
    optimizer2,
    T_max=CNN_EPOCHS,
    eta_min=1e-6
)

# ✅ NEW (mixed precision for speed)
scaler = torch.amp.GradScaler('cuda')

print(f"\nTraining CNN Tier-A — {CNN_EPOCHS} epochs")
print(f"  Images  : {len(cnn_train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

cnn_losses = []
cnn_accs   = []

# ================= TRAIN LOOP =================
for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in cnn_train_loader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)

        optimizer2.zero_grad()

        # ✅ mixed precision
        with torch.amp.autocast('cuda'):
            logits = cnn_model(images)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(cnn_model.parameters(), 1.0)

        scaler.step(optimizer2)
        scaler.update()

        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    scheduler2.step()

    avg_loss = total_loss / len(cnn_train_loader)
    acc = correct / total * 100

    cnn_losses.append(avg_loss)
    cnn_accs.append(acc)

    print(f"Epoch {epoch+1:02d}/{CNN_EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ CNN training complete")

# ================= EVALUATION =================
print("\nEvaluating CNN Tier-A...")
cnn_metrics = evaluate_topk(
    cnn_model, cnn_test_loader, device, is_cnn=True)

print("\n" + "=" * 55)
print("IMPROVED DAY 7 — CNN TIER-A RESULTS")
print("=" * 55)

for k, v in cnn_metrics.items():
    print(f"  {k:<20}: {v}")

# ================= SAVE =================
torch.save({
    'model_state_dict': cnn_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : cnn_metrics
}, f"{MODELS}/improved_cnn_tier_a.pt")

print(f"\n✓ CNN Tier-A model saved")

Building CNN Tier-A datasets...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


✓ CNN DataLoaders ready
  Train : 13698
  Test  : 3425
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 196MB/s]



Training CNN Tier-A — 10 epochs
  Images  : 13698
  Classes : 88
-------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# Final Summary + Save

In [ ]:
import matplotlib.pyplot as plt

# Plot curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(nlp_losses, color='#378ADD',
             linewidth=2, marker='o', markersize=3)
axes[0].set_title('NLP Loss — Tier-A 20 epochs')
axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.3)

axes[1].plot(cnn_losses, color='#1D9E75',
             linewidth=2, marker='o', markersize=3)
axes[1].set_title('CNN Loss — Tier-A 20 epochs')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS}/day7_improved_tier_a.png",
            dpi=150, bbox_inches='tight')
plt.show()

# Save summary
summary = {
    "day"        : "7_improved",
    "approach"   : "Tier-A (88 classes) 20 epochs",
    "num_classes": NUM_CLASSES,
    "nlp_metrics": nlp_metrics,
    "cnn_metrics": cnn_metrics,
    "vs_original": {
        "nlp_before": 16.89,
        "nlp_after" : nlp_metrics.get('accuracy', 0),
        "cnn_before": 10.20,
        "cnn_after" : cnn_metrics.get('accuracy', 0)
    },
    "status": "Day 7 improved complete"
}

with open(f"{RESULTS}/day7_improved_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 55)
print("DAY 7 IMPROVED COMPLETE ✓")
print("=" * 55)
print(f"  NLP: 16.89% → "
      f"{nlp_metrics.get('accuracy', '?')}%")
print(f"  CNN: 10.20% → "
      f"{cnn_metrics.get('accuracy', '?')}%")
print(f"\n  Models saved:")
print(f"    improved_nlp_tier_a.pt")
print(f"    improved_cnn_tier_a.pt")